# 7주차 실습: 이진 탐색 트리(BST) 구현 및 성능 분석 — 과제 완성본

## 목표·규약
재귀 탐색/삽입/삭제, 중위 정렬 검증, 삽입 순서별 높이·탐색 시간 비교.
Python 3.8+; 위에서 아래로 실행한다. 키는 비교 가능한 정수이고, 중복 삽입은 무시하며 없는 값 삭제는 변화가 없다.
높이는 간선 기준이며 빈 트리는 -1이다. 단순 BST는 균형을 보장하지 않는다.
큰 편향 트리는 반복 삽입·반복 탐색·반복 높이 계산으로 처리한다.


In [1]:
import random, time, statistics
from collections import deque
print("환경 준비 완료")

환경 준비 완료


## Step 1. BSTNode
지난주 TreeNode의 data/left/right 구조를 재사용.

In [2]:
class BSTNode:
    def __init__(self, data):
        self.data = data
        self.left = None
        self.right = None
    def __repr__(self):
        return f"BSTNode({self.data!r})"

## Step 2–3. 전체 BST 클래스
재귀 결과를 `node.left/right`, 최상위 결과를 `self.root`에 재연결하세요.
삭제: 리프→None / 자식 하나→자식 / 둘→오른쪽 최소값 복사 후 후속자 삭제.
후속자는 오른쪽 자식을 가질 수 있어 재연결이 필요합니다.
하나의 클래스에 전체 메서드를 넣어 동일 이름 자기 상속을 피했습니다.
성능 실험용 반복 메서드도 포함합니다. 큰 편향 트리에 재귀 메서드를 호출하지 마세요.


In [3]:
class BinarySearchTree:
    def __init__(self):
        self.root = None

    def search(self, key):
        return self._search(self.root, key)

    def _search(self, node, key):
        if node is None:
            return False
        if key == node.data:
            return True
        if key < node.data:
            return self._search(node.left, key)
        return self._search(node.right, key)

    def insert(self, key):
        self.root = self._insert(self.root, key)

    def _insert(self, node, key):
        if node is None:
            return BSTNode(key)
        if key < node.data:
            node.left = self._insert(node.left, key)
        elif key > node.data:
            node.right = self._insert(node.right, key)
        return node

    def delete(self, key):
        self.root = self._delete(self.root, key)

    def _delete(self, node, key):
        if node is None:
            return None
        if key < node.data:
            node.left = self._delete(node.left, key)
        elif key > node.data:
            node.right = self._delete(node.right, key)
        else:
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left
            successor = self._find_min(node.right)
            node.data = successor.data
            node.right = self._delete(node.right, successor.data)
        return node

    def _find_min(self, node):
        while node.left is not None:
            node = node.left
        return node

    def inorder(self):
        result = []

        def visit(node):
            if node is None:
                return
            visit(node.left)
            result.append(node.data)
            visit(node.right)
        visit(self.root)
        return result

    def height(self):
        def get_height(node):
            if node is None:
                return -1
            return 1 + max(
                get_height(node.left),
                get_height(node.right)
            )
        return get_height(self.root)

    def count_nodes(self):
        def count(node):
            if node is None:
                return 0
            return 1 + count(node.left) + count(node.right)
        return count(self.root)

    def insert_iterative(self, key):
        if self.root is None:
            self.root = BSTNode(key)
            return
        node = self.root
        while True:
            if key < node.data:
                if node.left is None:
                    node.left = BSTNode(key)
                    return
                node = node.left
            elif key > node.data:
                if node.right is None:
                    node.right = BSTNode(key)
                    return
                node = node.right
            else:
                return

    def search_iterative(self, key):
        node = self.root
        while node is not None:
            if key == node.data:
                return True
            if key < node.data:
                node = node.left
            else:
                node = node.right
        return False

    def height_iterative(self):
        if self.root is None:
            return -1
        stack = [(self.root, 0)]
        max_height = 0
        while stack:
            node, depth = stack.pop()
            max_height = max(max_height, depth)
            if node.left is not None:
                stack.append((node.left, depth + 1))
            if node.right is not None:
                stack.append((node.right, depth + 1))
        return max_height



## Step 4. 삽입/삭제 로직 검증
후속자의 오른쪽 자식, 루트 삭제, 중복, 없는 값, 무작위 혼합 연산까지 검사.

In [4]:
def make_bst(values, iterative=False):
    tree = BinarySearchTree()
    for value in values:
        if iterative: tree.insert_iterative(value)
        else: tree.insert(value)
    return tree

def assert_bst(tree, expected):
    # 조상 범위까지 검사: 바로 위 부모만 비교하는 오류 방지
    def valid(node, low=None, high=None):
        if node is None: return True
        if low is not None and node.data <= low: return False
        if high is not None and node.data >= high: return False
        return valid(node.left, low, node.data) and valid(node.right, node.data, high)
    assert valid(tree.root)
    assert tree.inorder() == sorted(set(expected))
    assert tree.count_nodes() == len(set(expected))


In [5]:
values = [50, 30, 70, 20, 40, 60, 80]
bst = make_bst(values)
assert_bst(bst, values)
assert bst.search(40) is True and bst.search(99) is False
for key in [20, 30, 70]:
    bst.delete(key)
    values.remove(key)
    assert_bst(bst, values)
    assert not bst.search(key)
    print("삭제", key, bst.inorder())
# 루트 두 자식 삭제
bst.delete(50); values.remove(50)
assert_bst(bst, values)
assert bst.root.data == 60

empty = BinarySearchTree()
empty.delete(1)
assert_bst(empty, [])
assert empty.height() == -1 and not empty.search(1)
for vals in [[5], [5,3], [5,7]]:
    t = make_bst(vals)
    t.delete(5)
    assert_bst(t, vals[1:])
# 후속자 60은 오른쪽 자식65를 갖는다: 65가 손실되면 안 됨.
t = make_bst([50,30,70,60,65,80])
t.delete(50)
assert t.root.data == 60 and t.root.right.left.data == 65
assert_bst(t, [30,60,65,70,80])
t.insert(65); t.delete(999)
assert_bst(t, [30,60,65,70,80])
# 전체 삭제로 빈 트리 복귀
for key in [30,60,65,70,80]: t.delete(key)
assert_bst(t, [])
# 고정 시드의 혼합 삽입/삭제를 Python set과 대조
rng = random.Random(7)
t, reference = BinarySearchTree(), set()
for _ in range(200):
    key = rng.randrange(50)
    if rng.random() < 0.6: t.insert(key); reference.add(key)
    else: t.delete(key); reference.discard(key)
    assert_bst(t, reference)
    assert t.search(key) == (key in reference)
print("모든 삽입·삭제·경계·불변식 테스트 통과")


삭제 20 [30, 40, 50, 60, 70, 80]
삭제 30 [40, 50, 60, 70, 80]
삭제 70 [40, 50, 60, 80]
모든 삽입·삭제·경계·불변식 테스트 통과


## Step 5. 높이 비교 실험
n=1000 재귀 삽입은 편향 트리에서 RecursionError가 날 수 있습니다. 반복 구현을 이용하며 재귀 제한을 올리지 않습니다.
최소 가능 높이 n>=1: ceil(log2(n+1))-1 = n.bit_length()-1.
무작위 트리는 기대 높이 O(log n)이지만 매번 균형이 보장되는 것은 아닙니다.


In [6]:
def build_for_experiment(n, order="random", seed=7):
    values = list(range(n))
    if order == "random":
        random.Random(seed).shuffle(values)
    elif order != "sorted":
        raise ValueError("order는 'random' 또는 'sorted'여야 합니다.")
    tree = BinarySearchTree()
    for value in values:
        tree.insert_iterative(value)
    return tree

def measure_height(n, order="random", seed=7):
    return build_for_experiment(n, order, seed).height_iterative()

n = 1000
bst_random = build_for_experiment(n, "random")
bst_sorted = build_for_experiment(n, "sorted")
assert bst_sorted.height_iterative() == n-1
print("최소 가능 높이:", n.bit_length()-1)
print("무작위/정렬 높이:", bst_random.height_iterative(), bst_sorted.height_iterative())


최소 가능 높이: 9
무작위/정렬 높이: 24 999


## Step 6. 탐색 시간 측정
동일한 키 목록, 반복 탐색 구현, 구성 시간 제외, 워밍업, 5회 중앙값. 측정값은 환경에 따라 달라집니다.

In [7]:
targets = [random.Random(17+i).randrange(n) for i in range(1000)]
def timed_search(tree, targets, repeats=5):
    for key in targets:
        tree.search_iterative(key)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        for key in targets:
            tree.search_iterative(key)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    return statistics.median(times)
random_time = timed_search(bst_random, targets)
sorted_time = timed_search(bst_sorted, targets)
print(f"무작위 트리 탐색 중앙값: {random_time:.6f}s")
print(f"정렬 삽입 트리 중앙값: {sorted_time:.6f}s")
if random_time > 0: print(f"측정 시간비: {sorted_time/random_time:.2f}")

무작위 트리 탐색 중앙값: 0.001121s
정렬 삽입 트리 중앙값: 0.025624s
측정 시간비: 22.86


## 과제: BST 구현 및 성능 분석
1. 모든 로직 테스트 통과. 2. n=100/1000/10000의 두 삽입 순서 높이를 표로 정리.
3. 최소 높이와 비교하고 성능 영향을 3~5문장으로 분석. 시간 측정 조건도 명시.
4. 개인 GitHub Week7 폴더, 주당 최소 2회 커밋, 과제 폼 GitHub 링크 제출.
**주의:** 정렬 순서 실제 삽입은 전체 O(n²)입니다. n=10000 셀은 느릴 수 있어 마지막에 실행합니다. 체인 직접 구성은 같은 삽입 실험이 아니므로 대체하지 않습니다.


In [8]:
# 입력 크기별 실제 삽입과 탐색 실험.
# n=1000은 위에서 만든 트리와 측정값을 재사용한다.
# 정렬 입력도 insert_iterative로 실제 삽입하며 체인 직접 구성은 하지 않는다.
experiment_rows = []
for size in [100, 1000, 10000]:
    print(f"n={size} 실험 중...", flush=True)
    if size == n:
        random_tree, sorted_tree = bst_random, bst_sorted
        current_targets = targets
        random_seconds, sorted_seconds = random_time, sorted_time
    else:
        random_tree = build_for_experiment(size, "random", seed=7)
        sorted_tree = build_for_experiment(size, "sorted", seed=7)
        current_targets = [random.Random(17+i).randrange(size) for i in range(1000)]
        random_seconds = timed_search(random_tree, current_targets, repeats=5)
        sorted_seconds = timed_search(sorted_tree, current_targets, repeats=5)

    h_random = random_tree.height_iterative()
    h_sorted = sorted_tree.height_iterative()
    min_height = size.bit_length() - 1
    assert h_sorted == size - 1
    assert h_random >= min_height
    assert h_sorted >= min_height
    assert all(random_tree.search_iterative(key) for key in current_targets)
    assert all(sorted_tree.search_iterative(key) for key in current_targets)
    experiment_rows.append({
        "n": size,
        "min_height": min_height,
        "random_height": h_random,
        "sorted_height": h_sorted,
        "random_seconds": random_seconds,
        "sorted_seconds": sorted_seconds,
        "time_ratio": sorted_seconds / random_seconds if random_seconds else None,
    })

rows = [(r["n"], r["min_height"], r["random_height"], r["sorted_height"])
        for r in experiment_rows]
print("\nn | 최소 가능 높이 | 무작위 높이 | 정렬 높이")
for row in rows:
    print(*row, sep=" | ")
print("\nn | 무작위 탐색 중앙값(ms) | 정렬 탐색 중앙값(ms) | 시간비")
for r in experiment_rows:
    ratio = f'{r["time_ratio"]:.2f}' if r["time_ratio"] is not None else "측정 불가"
    print(f'{r["n"]} | {r["random_seconds"]*1000:.6f} | '
          f'{r["sorted_seconds"]*1000:.6f} | {ratio}')
print("전체 크기별 높이 및 탐색 검증 통과")


n=100 실험 중...
n=1000 실험 중...
n=10000 실험 중...

n | 최소 가능 높이 | 무작위 높이 | 정렬 높이
100 | 6 | 12 | 99
1000 | 9 | 24 | 999
10000 | 13 | 29 | 9999

n | 무작위 탐색 중앙값(ms) | 정렬 탐색 중앙값(ms) | 시간비
100 | 0.423194 | 2.133031 | 5.04
1000 | 1.120874 | 25.623600 | 22.86
10000 | 1.919009 | 307.738124 | 160.36
전체 크기별 높이 및 탐색 검증 통과


## 실험 결과

| n | 최소 가능 높이 | 무작위 높이 | 정렬 높이 |
|---:|---:|---:|---:|
| 100 | 6 | 12 | 99 |
| 1,000 | 9 | 24 | 999 |
| 10,000 | 13 | 29 | 9,999 |

| n | 무작위 탐색 중앙값(ms) | 정렬 탐색 중앙값(ms) | 정렬/무작위 시간비 |
|---:|---:|---:|---:|
| 100 | 0.423194 | 2.133031 | 5.04배 |
| 1,000 | 1.120874 | 25.623600 | 22.86배 |
| 10,000 | 1.919009 | 307.738124 | 160.36배 |

## 성능 분석: 5문장

n=100·1,000·10,000에서 최소 가능 높이는 각각 6·9·13이고, 무작위 삽입 높이는 12·24·29, 정렬 삽입 높이는 99·999·9,999로 측정되었다. 최소 높이는 노드를 각 레벨에 최대한 채운 경우의 하한이며, 일반 BST의 무작위 삽입은 회전 등으로 균형을 강제하지 않으므로 이보다 높아질 수 있다. 오름차순 삽입에서는 모든 새 키가 기존 키보다 커 오른쪽 사슬이 생기므로 높이가 n−1이 되고, 탐색의 최악 시간은 O(n), 전체 트리 구성 시간은 O(n²)가 된다. 각 크기에서 동일한 1,000개 성공 탐색 대상을 사용한 결과, n=10,000의 무작위·정렬 탐색 중앙값은 각각 1.919009ms·307.738124ms로 정렬 삽입 트리가 약 160.36배 오래 걸렸다. 무작위 순열 삽입의 기대 연산 시간은 O(log n)이지만 최악의 균형을 보장하지 않으며, 이 실험의 시간비는 고정된 입력·실행 환경에서의 관측값이므로 이론적 시간 복잡도 자체나 모든 환경의 성능비를 뜻하지 않는다.

## 시간 측정 조건

| 항목 | 조건 |
|---|---|
| 실행 환경 | Python 3.12.14 / Linux x86_64; 사용자의 Colab이 아닌 검증용 Python 환경 |
| 입력 크기 | n = 100, 1,000, 10,000 |
| 키 집합 | 각 크기에서 양쪽 모두 list(range(n)); 중복 없는 같은 키 집합 |
| 무작위 삽입 | random.Random(7).shuffle(values) |
| 정렬 삽입 | 오름차순으로 insert_iterative를 실제 n회 호출 |
| 탐색 대상 | [random.Random(17+i).randrange(n) for i in range(1000)]; 타깃 중복 가능 |
| 탐색 유형 | 모두 존재하는 키의 성공 탐색; 같은 크기의 두 트리에 같은 목록 사용 |
| 측정 구간 | search_iterative로 1,000개 목록 전체를 탐색하는 시간; 단일 탐색 시간이 아님 |
| 워밍업·반복 | 목록 전체 1회 워밍업 후 5회 측정한 중앙값 |
| 타이머 | time.perf_counter() |
| 측정 제외 | 트리 구성, 타깃 목록 생성, 높이 계산, 검증 assert, 출력 |
| 재귀 제한 | 변경하지 않음; 큰 트리 삽입·탐색·높이는 반복 방식 사용 |

재실행하면 시간은 달라질 수 있다. 고정 시드의 높이는 같은 Python 셔플 결과를 사용하는 환경에서 재현할 수 있다.

## 복잡도 정리

n은 노드 수, h는 간선 기준 높이이다. 재귀 삭제의 후속자 탐색과 제거는 각각 트리 높이 이하이므로 합쳐도 O(h+1)이다.

| 연산 | 시간 | 보조 공간: 반환 결과 제외 |
|---|---|---|
| 재귀 탐색·삽입·삭제 | O(h+1) | O(h+1) |
| 반복 탐색·삽입 | O(h+1) | O(1); 삽입으로 만드는 새 노드 공간은 별도 |
| 중위 순회 | O(n) | O(h+1); 반환 리스트는 별도 O(n) |
| 재귀 높이·노드 수 | O(n) | O(h+1) |
| 반복 높이 계산 | O(n) | O(h+1) 상한; 오른쪽 사슬에서는 O(1) |
| 전체 BST 구성: 무작위 순열 | 기대 O(n log n), 최악 O(n²) | 생성 노드 O(n) |
| 전체 BST 구성: 정렬된 서로 다른 키 | O(n²) | 생성 노드 O(n) |
| Tree Sort | 기대 O(n log n), 최악 O(n²) | 트리·출력 O(n) |

현재 중복 삽입을 무시하므로 중복을 보존하는 정렬이 필요하면 빈도 카운터 등의 정책이 필요하다. 무작위 순열의 기대 높이는 O(log n)이지만 일반 BST는 최악의 높이 O(n)을 가질 수 있다. 최소 높이는 n>=1에서 ceil(log2(n+1))−1 = n.bit_length()−1이며, 빈 트리는 별도 규약으로 -1이다.


## 선택 심화·시각화
- VisuAlgo BST 모드에서 50,30,70,20,40,60,80 삽입 후 20,30,70 삭제와 코드 결과 비교.
- 중위 선행자를 이용한 삭제로 변경.
- 셔플 전처리는 기대 성능 개선이지 최악 균형 보장이 아님을 설명.
- 자손 수 저장으로 kth_smallest 구현: O(h), 균형일 때만 O(log n). 삽입/삭제마다 size 갱신 필요.
- 다음 주 중간 평가: 연결리스트, 스택/큐, 트리, BST 복습.
